# Q5 — Twitter Bot Detection

**Task:** Build a binary classifier to distinguish bot accounts from genuine users using Twitter profile metadata.

**Metric:** AUC (area under ROC curve). Submission is predicted probabilities.

> **TODO (write in your own words):** Explain in 2-3 sentences why AUC is the right metric here (think about class imbalance, threshold-free ranking, what the problem actually asks).


## Setup

In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore", category=UserWarning)
import re

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2025
DATA_DIR = Path("cs-610-assignment-1-question-5-2026")
SUB_DIR = Path("submissions"); SUB_DIR.mkdir(exist_ok=True)


## Load and inspect

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
y_train = train["target"].values

print(f"train: {train.shape}, test: {test.shape}")
print(f"bot rate: {y_train.mean():.3f}")
train.head()


In [ ]:
pd.DataFrame({
    "dtype": train.dtypes,
    "missing": train.isna().sum(),
    "pct": (train.isna().mean()*100).round(2),
    "nunique": train.nunique(),
})


> **TODO (write in your own words):** Note observations — class balance, missingness, which columns are unique-per-row (and so useless raw), anything else relevant.


## Feature engineering

> **TODO (write in your own words):** Justify the engineering: log-transforms for heavy-tailed counts, top-8 lang bucketing, presence flags alongside content, cyclical sin/cos for hour-of-day, ratio features.


In [ ]:
TOP_LANGS = train["lang"].value_counts().head(8).index.tolist()

EMOJI_RE = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F\U0001F1E0-\U0001F1FF\U00002700-\U000027BF"
    "\U00002600-\U000026FF]"
)

def featurize(df):
    out = pd.DataFrame(index=df.index)

    # counts + log1p (heavy-tailed)
    for c in ["favourites_count","followers_count","friends_count",
              "statuses_count","average_tweets_per_day","account_age_days"]:
        out[c] = df[c].astype(float)
    for c in ["favourites_count","followers_count","friends_count","statuses_count"]:
        out[f"log_{c}"] = np.log1p(out[c])

    # booleans
    for c in ["default_profile","default_profile_image","geo_enabled","verified"]:
        out[c] = df[c].astype(int)

    # presence flags
    out["has_description"] = df["description"].notna().astype(int)
    out["has_location"] = (df["location"].notna() & (df["location"] != "unknown")).astype(int)
    out["has_url_bg"] = df["profile_background_image_url"].notna().astype(int)

    # text length / pattern
    out["desc_len"] = df["description"].fillna("").str.len()
    out["sn_len"] = df["screen_name"].fillna("").str.len()
    out["sn_digits"] = (df["screen_name"].fillna("").str.count(r"\d") / out["sn_len"].clip(lower=1))

    # lang bucket
    lang = df["lang"].fillna("MISSING")
    out["lang"] = lang.where(lang.isin(TOP_LANGS), "OTHER")

    # ratios
    out["followers_per_friend"] = df["followers_count"] / df["friends_count"].clip(lower=1)
    out["statuses_per_day"] = df["statuses_count"] / df["account_age_days"].clip(lower=1)

    # temporal
    ct = pd.to_datetime(df["created_at"])
    out["created_year"] = ct.dt.year.astype(float)
    out["created_month"] = ct.dt.month.astype(float)
    out["created_dow"] = ct.dt.dayofweek.astype(float)
    out["created_hour"] = ct.dt.hour.astype(float)
    out["hour_sin"] = np.sin(2*np.pi*out["created_hour"]/24)
    out["hour_cos"] = np.cos(2*np.pi*out["created_hour"]/24)
    out["dow_sin"] = np.sin(2*np.pi*out["created_dow"]/7)
    out["dow_cos"] = np.cos(2*np.pi*out["created_dow"]/7)

    # bio structure (writing style)
    s = df["description"].fillna("")
    n = s.str.len().clip(lower=1)
    out["bio_url_count"] = s.str.count(r"https?://")
    out["bio_hashtag_count"] = s.str.count(r"#\w+")
    out["bio_mention_count"] = s.str.count(r"@\w+")
    out["bio_emoji_count"] = s.apply(lambda x: len(EMOJI_RE.findall(x)))
    out["bio_caps_ratio"] = s.str.count(r"[A-Z]") / n
    out["bio_digit_ratio"] = s.str.count(r"\d") / n
    out["bio_special_ratio"] = s.str.count(r"[^\w\s]") / n
    out["bio_word_count"] = s.str.split().apply(len)
    return out

X_train = featurize(train)
X_test = featurize(test)
print("feature shape:", X_train.shape)


### TF-IDF on description

> **TODO (write in your own words):** Why TF-IDF? Why fit on train only (leakage)? Why these parameters specifically?


In [ ]:
tfidf = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2),
    min_df=5, max_df=0.95,
    lowercase=True, sublinear_tf=True, strip_accents="unicode",
)
Xt_train = tfidf.fit_transform(train["description"].fillna(""))
Xt_test = tfidf.transform(test["description"].fillna(""))
print("tf-idf:", Xt_train.shape)


### Assemble final feature matrix

In [ ]:
numeric_cols = [c for c in X_train.columns if c != "lang"]

oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
lang_train_oh = oh.fit_transform(X_train[["lang"]])
lang_test_oh = oh.transform(X_test[["lang"]])

num_train = sparse.csr_matrix(X_train[numeric_cols].fillna(0).values)
num_test = sparse.csr_matrix(X_test[numeric_cols].fillna(0).values)

X_sp_train = sparse.hstack([num_train, lang_train_oh, Xt_train]).tocsr()
X_sp_test = sparse.hstack([num_test, lang_test_oh, Xt_test]).tocsr()
print("combined:", X_sp_train.shape)


## 5-fold CV helper

> **TODO (write in your own words):** Why CV over a single split? Why stratified? What does the OOF prediction give us that fold-by-fold AUCs don't?


In [ ]:
def lgbm_cv(X, y, X_test, n_splits=5, params=None, verbose=True):
    p = dict(n_estimators=2000, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE)
    if params: p.update(params)
    take = lambda M, idx: M.iloc[idx] if hasattr(M, "iloc") else M[idx]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, iters = [], []
    oof = np.zeros(X.shape[0])
    test_preds = np.zeros((n_splits, X_test.shape[0]))
    for fold, (tr, va) in enumerate(skf.split(np.zeros(X.shape[0]), y)):
        clf = lgb.LGBMClassifier(**p)
        clf.fit(take(X, tr), y[tr],
                eval_set=[(take(X, va), y[va])], eval_metric="auc",
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        oof[va] = clf.predict_proba(take(X, va))[:, 1]
        test_preds[fold] = clf.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        iters.append(clf.best_iteration_)
        if verbose: print(f"fold {fold+1}: AUC={aucs[-1]:.4f} iter={iters[-1]}")
    if verbose:
        print(f"OOF AUC: {roc_auc_score(y, oof):.4f} (mean {np.mean(aucs):.4f} +/- {np.std(aucs):.4f})")
    return {"oof": oof, "test_avg": test_preds.mean(axis=0), "aucs": aucs, "iters": iters}


## Model 1: LightGBM (Optuna-tuned)

> **TODO (write in your own words):** Why LightGBM (vs RF / LogReg)? One sentence on what Optuna does. Which hyperparameters mattered most in the search (look at your importance chart).


In [ ]:
# best params from 50-trial optuna run targeting OOF AUC
best_params = {
    "learning_rate":    0.0232,
    "num_leaves":       90,
    "min_data_in_leaf": 14,
    "feature_fraction": 0.5042,
    "bagging_fraction": 0.9708,
    "bagging_freq":     4,
    "lambda_l2":        3.437,
}

print("=== LightGBM ===")
lgb_result = lgbm_cv(X_sp_train, y_train, X_sp_test, params=best_params)
oof_lgb = lgb_result["oof"]
test_lgb = lgb_result["test_avg"]


## Model 2: CatBoost (for diversity)

> **TODO (write in your own words):** What does "diversity" mean in ensembling? Why CatBoost specifically (different from LightGBM in tree growth + categorical handling = uncorrelated mistakes).


In [ ]:
def cb_cv(X, y, X_test, n_splits=5, verbose=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    oof = np.zeros(len(X))
    test_preds = np.zeros((n_splits, len(X_test)))
    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(X)), y)):
        clf = CatBoostClassifier(
            iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
            loss_function="Logloss", eval_metric="AUC",
            random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
        )
        clf.fit(X.iloc[tr], y[tr], eval_set=(X.iloc[va], y[va]),
                cat_features=["lang"], use_best_model=True)
        oof[va] = clf.predict_proba(X.iloc[va])[:, 1]
        test_preds[fold] = clf.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        if verbose: print(f"fold {fold+1}: AUC={aucs[-1]:.4f}")
    if verbose:
        print(f"OOF AUC: {roc_auc_score(y, oof):.4f} (mean {np.mean(aucs):.4f} +/- {np.std(aucs):.4f})")
    return {"oof": oof, "test_avg": test_preds.mean(axis=0)}

print("=== CatBoost ===")
cb_result = cb_cv(X_train, y_train, X_test)
oof_cb = cb_result["oof"]
test_cb = cb_result["test_avg"]
print(f"\ncorr(LGB, CB) = {np.corrcoef(oof_lgb, oof_cb)[0,1]:.4f}")


## Model 3: XGBoost (third base model)

> **TODO (write in your own words):** Why add a third model? Note that XGBoost is also a GBM but with different tree-growth (level-wise vs LightGBM's leaf-wise) — different mistakes, more diversity for the blend.


In [ ]:
def xgb_cv(X, y, X_test, n_splits=5, verbose=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    oof = np.zeros(X.shape[0])
    test_preds = np.zeros((n_splits, X_test.shape[0]))
    for fold, (tr, va) in enumerate(skf.split(np.zeros(X.shape[0]), y)):
        clf = xgb.XGBClassifier(
            n_estimators=2000, learning_rate=0.05, max_depth=6,
            min_child_weight=5, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, tree_method="hist", eval_metric="auc",
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
            early_stopping_rounds=50,
        )
        clf.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = clf.predict_proba(X[va])[:, 1]
        test_preds[fold] = clf.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y[va], oof[va]))
        if verbose: print(f"fold {fold+1}: AUC={aucs[-1]:.4f}")
    if verbose:
        print(f"OOF AUC: {roc_auc_score(y, oof):.4f} (mean {np.mean(aucs):.4f} +/- {np.std(aucs):.4f})")
    return {"oof": oof, "test_avg": test_preds.mean(axis=0)}

print("=== XGBoost ===")
xgb_result = xgb_cv(X_sp_train, y_train, X_sp_test)
oof_xgb = xgb_result["oof"]
test_xgb = xgb_result["test_avg"]

print(f"\ncorr(LGB, XGB) = {np.corrcoef(oof_lgb, oof_xgb)[0,1]:.4f}")
print(f"corr(CB,  XGB) = {np.corrcoef(oof_cb,  oof_xgb)[0,1]:.4f}")


## Stacking — learn the optimal blend weights

> **TODO (write in your own words):** Why use a meta-model instead of fixed weights? What is the meta-model actually learning? Note that we train it on OOF predictions to avoid leakage (otherwise the LR would just memorize where each base model overfit).


In [ ]:
# Meta-model trained on OOF predictions of the 3 base models
oof_stack = np.column_stack([oof_lgb, oof_cb, oof_xgb])
test_stack = np.column_stack([test_lgb, test_cb, test_xgb])

# Logistic regression as a simple linear blender
# (constrains weights to sum to ~1 in the right direction via the sigmoid)
meta = LogisticRegression(C=1.0, max_iter=1000)
meta.fit(oof_stack, y_train)
oof_meta = meta.predict_proba(oof_stack)[:, 1]
test_meta = meta.predict_proba(test_stack)[:, 1]

print(f"LightGBM     OOF: {roc_auc_score(y_train, oof_lgb):.4f}")
print(f"CatBoost     OOF: {roc_auc_score(y_train, oof_cb):.4f}")
print(f"XGBoost      OOF: {roc_auc_score(y_train, oof_xgb):.4f}")
print(f"Stacked meta OOF: {roc_auc_score(y_train, oof_meta):.4f}")
print(f"\nMeta coefficients (LGB, CB, XGB): {meta.coef_[0]}")


## Final submission

> **TODO (write in your own words):** Explain the final approach in 2-3 sentences. Mention that the submission is the stacked output, that everything was trained in this notebook, and how to interpret the meta-coefficients (which model the meta-LR weighted most heavily).


In [ ]:
sub = pd.DataFrame({"index": test["index"].values, "target": test_meta})
sub.to_csv(SUB_DIR / "final_submission.csv", index=False, float_format="%.6f")
print(f"wrote: {SUB_DIR/'final_submission.csv'}")
print(f"\nprediction summary:")
print(f"  mean: {test_meta.mean():.4f}  (train base rate {y_train.mean():.4f})")
print(f"  min:  {test_meta.min():.4f}")
print(f"  max:  {test_meta.max():.4f}")
sub.head()


## Results

| Stage | Model | OOF AUC |
|---|---|---|
| Single | LightGBM (Optuna-tuned) | (your number) |
| Single | CatBoost | (your number) |
| Single | XGBoost | (your number) |
| Stacked | Meta-LR over the three | **(your number)** |

> **TODO (write in your own words):** 4-5 bullets on what worked, what didn't. Examples: TF-IDF added clear lift; temporal features were tiny; Optuna lifted LightGBM by ~0.002; CatBoost gave structural diversity but lower correlation than expected; stacking found a useful weighting. Mention the consistent OOF -> public LB gap (~0.005) you observed as systematic distribution shift between train and test.
